In [1]:
!pip install dash
!pip install prophet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 4.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 3.8 MB/s eta 0:00:00a 0:00:01


In [19]:

import dash
from dash import html
from dash import dcc
from dash.dependencies import Input, Output
import yfinance as yf
import pandas as pd
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA

import numpy as np

app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Stock Price Forecasting App"),
    html.Label("Enter Stock Symbol:"),
    dcc.Input(id='stock-input', type='text', value='AAPL'),
    html.Label("\nEnter Forecast Period (Days):"),
    dcc.Input(id='period-input', type='number', value=30),
    html.Button('Submit', id='submit-val', n_clicks=0),
    html.Div(id='output-prediction')
])

@app.callback(
    Output('output-prediction', 'children'),
    Input('submit-val', 'n_clicks'),
    Input('stock-input', 'value'),
    Input('period-input', 'value')
)
def update_output(n_clicks, stock_input,period_input):
    stock_data = yf.download(stock_input, start='2021-01-01', end='2021-12-31')
    stock_data['Date'] = stock_data.index
    stock_data.reset_index(drop=True, inplace=True)
    
    # ARIMA model
    arima_model = ARIMA(stock_data['Close'], order=(5,1,0))
    arima_fit = arima_model.fit()
    arima_forecast = arima_fit.forecast(steps=period_input)
    
    # Prophet model
    prophet_data = stock_data[['Date', 'Close']]
    prophet_data.columns = ['ds', 'y']
    prophet_data['ds'] = prophet_data['ds'].dt.tz_localize(None)
    prophet_model = Prophet()
    prophet_model.fit(prophet_data)
    future = prophet_model.make_future_dataframe(periods=period_input)
    prophet_forecast = prophet_model.predict(future)

    val1=(arima_forecast).mean()
    val2=prophet_forecast['yhat'].mean()
  
    return html.Div([
        html.H2("ARIMA Forecast"),
        html.P(val1.round(2)),
        html.H2("Prophet Forecast"),
        html.P((val2.round(2))) #.tail(period_input))
      
    ])

if __name__ == '__main__':
    app.run(debug=True)
